In [1]:
import sys
import os
from pathlib import Path

# Walk upward until we find the folder that contains "src"
p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent

if not (p / "src").exists():
    raise RuntimeError("Could not find repo root (folder containing 'src').")

if str(p) not in sys.path:
    sys.path.insert(0, str(p))

print("Repo root set to:", p)


Repo root set to: /home/iauger/projects/dsci632-project


In [2]:
import os, subprocess
import sys, platform

print("python:", sys.executable)
print("platform:", platform.platform())
print("cwd:", os.getcwd())
print("home:", str(Path.home()))

assert "/home/" in str(Path.home()), "Not running in WSL home"
assert "mnt/c" not in os.getcwd().lower(), "CWD is on Windows mount; use ~/projects/... instead"
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("which java:", subprocess.check_output(["which","java"]).decode().strip())

python: /home/iauger/projects/dsci632-project/.venv/bin/python
platform: Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
cwd: /home/iauger/projects/dsci632-project/notebooks
home: /home/iauger
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
which java: /usr/lib/jvm/java-17-openjdk-amd64/bin/java


In [3]:
from src.config import load_settings 

s = load_settings()
print("ENV:", s.env)
print("RAW recipes:", s.raw_recipes_path)
print("RAW interactions:", s.raw_interactions_path)
print("PROCESSED:", s.processed_dir)
print("BRONZE:", s.bronze_dir)
print("SILVER:", s.silver_dir)
print("GOLD:", s.gold_dir)
print("RUN_ID:", s.features_run_id)
print("FEATURES DATASET",s.features_dataset_path)

ENV: local
RAW recipes: /home/iauger/projects/dsci632-project/data/raw/RAW_recipes.csv
RAW interactions: /home/iauger/projects/dsci632-project/data/raw/RAW_interactions.csv
PROCESSED: /home/iauger/projects/dsci632-project/data/processed
BRONZE: /home/iauger/projects/dsci632-project/data/processed/bronze
SILVER: /home/iauger/projects/dsci632-project/data/processed/silver
GOLD: /home/iauger/projects/dsci632-project/data/processed/gold
RUN_ID: 20260227_011552
FEATURES DATASET /home/iauger/projects/dsci632-project/data/processed/features/runs/20260227_011552/dataset.parquet


In [4]:
from src.spark.session import get_spark
    
spark = get_spark(app_name="testing-spark-session", debug=True)

print("spark.version:", spark.version)
print("spark.driver.host:", spark.conf.get("spark.driver.host", "NOT SET"))
print("spark.local.ip:", spark.conf.get("spark.local.ip", "NOT SET"))

your 131072x1 screen size is bogus. expect trouble
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 01:19:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark.version: 3.5.1
spark.driver.host: 127.0.0.1
spark.local.ip: 127.0.0.1


In [5]:
from src.config import load_settings
from src.spark.features.io import read_labeled_reviews, write_parquet, write_metrics, ensure_dirs
from src.spark.features.splits import assign_recipe_splits, build_splits_table

ensure_dirs(s)

df = read_labeled_reviews(spark, s)
df2 = assign_recipe_splits(df, train_frac=0.8, val_frac=0.1, test_frac=0.1, seed=42)

splits_df = build_splits_table(df2)
write_parquet(splits_df, s.features_splits_path)

# quick metrics
counts = splits_df.groupBy("split").count().toPandas().set_index("split")["count"].to_dict()
write_metrics(s, {"split_counts": counts})
counts

{'train': 5989, 'test': 767, 'val': 744}

In [6]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

# If you use your settings object:
from src.config import load_settings
from src.spark.session import get_spark

# Your modules
from src.spark.features.build_features import build_features
from src.spark.features.prototypes import PrototypeSpec, build_tag_centroids
from src.spark.features.similarity import SimilaritySpec, score_review_tag_similarity_long
from src.spark.features.io import read_feature_dataset, write_parquet  # if you have read_parquet helper

In [7]:
build_features(spark)  

26/02/27 01:19:18 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/02/27 01:19:46 WARN TaskSetManager: Stage 59 contains a task of very large size (1063 KiB). The maximum recommended task size is 1000 KiB.


In [8]:
df = spark.read.parquet(s.features_dataset_path)

expected = {"review_key", "split", "features"}
missing = expected - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

# label cols
label_cols = [c for c in df.columns if c.startswith("y_")]
assert label_cols, "No y_* label columns found"

df.select("split").distinct().show()
df.groupBy("split").count().show()

df.select("features").printSchema()

# null feature vectors
null_feat = df.filter(F.col("features").isNull()).count()
print("null features:", null_feat)

# label prevalence (quick)
df.select(*[F.avg(F.col(c).cast("double")).alias(c) for c in label_cols]).show(truncate=False)

+-----+
|split|
+-----+
|train|
| test|
|  val|
+-----+

+-----+-----+
|split|count|
+-----+-----+
|train| 5989|
| test|  767|
|  val|  744|
+-----+-----+

root
 |-- features: vector (nullable = true)

null features: 0
+------------------+-------------------+--------------------+
|y_delicious_tasty |y_ingredient_issue |y_bland_lacks_flavor|
+------------------+-------------------+--------------------+
|0.7814666666666666|0.07133333333333333|0.03933333333333333 |
+------------------+-------------------+--------------------+



In [9]:
p_spec = PrototypeSpec(
    split_col="split",
    train_split_value="train",
    features_col="features",
    label_prefix="y_",
)

centroids = build_tag_centroids(df, spec=p_spec)
centroids.show(truncate=False)
centroids.printSchema()

26/02/27 01:19:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [10]:
# Should have one row per tag (minus any tags with 0 positives)
centroids.count(), len(label_cols)
centroids.select("tag").distinct().count()

# Check for null centroids
centroids.filter(F.col("centroid").isNull()).show()

# Check for very small pos_count tags (expected, but good to know)
centroids.orderBy(F.col("pos_count").asc()).show(20, truncate=False)

+---+---------+--------+
|tag|pos_count|centroid|
+---+---------+--------+
+---+---------+--------+

+------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
s_spec = SimilaritySpec(
    review_id_col="review_key",
    split_col="split",
    review_vec_col="features",
    centroid_tag_col="tag",
    centroid_vec_col="centroid",
    out_score_col="sim",
)

sim_val = score_review_tag_similarity_long(
    reviews_df=df,
    centroids_df=centroids,
    spec=s_spec,
    filter_split="val",
    broadcast_centroids=True,
)

sim_val.cache()
sim_val.show(10, truncate=False)
sim_val.groupBy("tag").agg(
    F.count("*").alias("n"),
    F.avg("sim").alias("avg_sim"),
    F.min("sim").alias("min_sim"),
    F.max("sim").alias("max_sim"),
).orderBy(F.desc("n")).show(50, truncate=False)

+----------------------------------------------------------------+-----+------------------+------------------+
|review_key                                                      |split|tag               |sim               |
+----------------------------------------------------------------+-----+------------------+------------------+
|728d6ae45a212ed975b16f6190b4b5bd556617de1c909d085f835f2bd83781f7|val  |delicious_tasty   |0.6285482374035477|
|728d6ae45a212ed975b16f6190b4b5bd556617de1c909d085f835f2bd83781f7|val  |ingredient_issue  |0.7437282640708804|
|728d6ae45a212ed975b16f6190b4b5bd556617de1c909d085f835f2bd83781f7|val  |bland_lacks_flavor|0.6739367756342279|
|bd1dffc9139753a9c61a137c69b1f979b4da227aae970ea1b7c246acae41fe8b|val  |delicious_tasty   |0.7532302589706178|
|bd1dffc9139753a9c61a137c69b1f979b4da227aae970ea1b7c246acae41fe8b|val  |ingredient_issue  |0.8103416991737072|
|bd1dffc9139753a9c61a137c69b1f979b4da227aae970ea1b7c246acae41fe8b|val  |bland_lacks_flavor|0.7643330117124425|
|

In [12]:
n_val = df.filter(F.col("split") == "val").count()
n_tags = centroids.count()
n_sim = sim_val.count()

print("n_val_reviews:", n_val)
print("n_tags:", n_tags)
print("n_sim_rows:", n_sim)
print("expected ~", n_val * n_tags)

n_val_reviews: 744
n_tags: 3
n_sim_rows: 2232
expected ~ 2232


In [13]:
# Create y value dynamically for each (review, tag) row
# y_col = concat("y_", tag)
sim_val_labeled = (
    sim_val
    .join(df.select("review_key", *label_cols), on="review_key", how="left")
    .withColumn("y_col", F.concat(F.lit("y_"), F.col("tag")))
    .withColumn("y", F.col("y_col"))  # placeholder
)

# You can't directly do F.col(dynamic_name) without a workaround.
# Workaround: build a map from label cols -> values.
label_map_expr = F.create_map(*sum([[F.lit(c.replace("y_", "")), F.col(c).cast("int")] for c in label_cols], []))

sim_val_labeled = (
    sim_val
    .join(df.select("review_key", label_map_expr.alias("y_map")), on="review_key", how="left")
    .withColumn("y", F.coalesce(F.col("y_map")[F.col("tag")], F.lit(0)))
    .drop("y_map")
)

sim_val_labeled.select("review_key", "tag", "sim", "y").show(20, truncate=False)
sim_val_labeled.groupBy("tag").agg(
    F.avg(F.col("y").cast("double")).alias("pos_rate"),
    F.count("*").alias("rows"),
).orderBy(F.desc("pos_rate")).show(50, truncate=False)

+----------------------------------------------------------------+------------------+------------------+---+
|review_key                                                      |tag               |sim               |y  |
+----------------------------------------------------------------+------------------+------------------+---+
|728d6ae45a212ed975b16f6190b4b5bd556617de1c909d085f835f2bd83781f7|delicious_tasty   |0.6285482374035477|1  |
|728d6ae45a212ed975b16f6190b4b5bd556617de1c909d085f835f2bd83781f7|ingredient_issue  |0.7437282640708804|0  |
|728d6ae45a212ed975b16f6190b4b5bd556617de1c909d085f835f2bd83781f7|bland_lacks_flavor|0.6739367756342279|0  |
|bd1dffc9139753a9c61a137c69b1f979b4da227aae970ea1b7c246acae41fe8b|delicious_tasty   |0.7532302589706178|1  |
|bd1dffc9139753a9c61a137c69b1f979b4da227aae970ea1b7c246acae41fe8b|ingredient_issue  |0.8103416991737072|0  |
|bd1dffc9139753a9c61a137c69b1f979b4da227aae970ea1b7c246acae41fe8b|bland_lacks_flavor|0.7643330117124425|0  |
|7c0e75a52c6b485525

In [14]:
test_tags = [row["tag"] for row in centroids.orderBy(F.desc("pos_count")).limit(3).collect()]

for t in test_tags:
    tmp = sim_val_labeled.filter(F.col("tag") == t)
    stats = tmp.groupBy("y").agg(
        F.count("*").alias("n"),
        F.avg("sim").alias("avg_sim"),
        F.expr("percentile_approx(sim, 0.5)").alias("median_sim"),
    )
    print("Tag:", t)
    stats.show(truncate=False)

Tag: delicious_tasty
+---+---+------------------+------------------+
|y  |n  |avg_sim           |median_sim        |
+---+---+------------------+------------------+
|1  |587|0.6447540361290313|0.6636815281255761|
|0  |157|0.6392088840185163|0.6449997218937831|
+---+---+------------------+------------------+

Tag: ingredient_issue
+---+---+------------------+------------------+
|y  |n  |avg_sim           |median_sim        |
+---+---+------------------+------------------+
|1  |53 |0.7035558589909371|0.7191753407269789|
|0  |691|0.6172250942234797|0.6420232254809752|
+---+---+------------------+------------------+

Tag: bland_lacks_flavor
+---+---+------------------+------------------+
|y  |n  |avg_sim           |median_sim        |
+---+---+------------------+------------------+
|1  |27 |0.7248442369641542|0.7499791544067705|
|0  |717|0.5915096427925104|0.6112747865447012|
+---+---+------------------+------------------+



In [ ]:
from src.config import load_settings
s = load_settings()
print(s.features_run_id)

In [ ]:
from src.spark.modeling.train import train_all
train_all(spark, settings=s)

In [ ]:
import shutil
from pathlib import Path

targets = [
    Path(s.bronze_recipes_path),
    Path(s.bronze_interactions_path),
    Path(s.silver_recipes_path),
    Path(s.silver_interactions_path),
    Path(s.gold_ve_path),
    Path(s.gold_cf_path),
]

for t in targets:
    if t.exists():
        shutil.rmtree(t)
        print("deleted:", t)
    else:
        print("missing (ok):", t)

In [ ]:
recipes_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("enforceSchema", "false")
    .csv(s.raw_recipes_path)
)

interactions_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("enforceSchema", "false")
    .csv(s.raw_interactions_path)
)

recipes_raw.write.mode("overwrite").parquet(s.bronze_recipes_path)
interactions_raw.write.mode("overwrite").parquet(s.bronze_interactions_path)

In [ ]:
recipes_b = spark.read.parquet(s.bronze_recipes_path)

# id should be numeric-ish and definitely not long sentences
recipes_b.select("id", "nutrition").show(10, truncate=False)

# sanity: find “bad id” rows (contains spaces or punctuation typical of text)
from pyspark.sql import functions as F
bad = recipes_b.filter(F.col("id").rlike(r"[A-Za-z]|\s"))
bad.select("id", "nutrition").show(20, truncate=False)
print("bad id count:", bad.count())

In [ ]:
from src.spark.ingestion.spark.transforms.recipes_spark import clean_recipes     
from src.spark.ingestion.spark.transforms.interactions_spark import clean_interactions 

recipes_b = spark.read.parquet(s.bronze_recipes_path)
interactions_b = spark.read.parquet(s.bronze_interactions_path)

recipes_s = clean_recipes(recipes_b)
interactions_s = clean_interactions(interactions_b)

recipes_s.write.mode("overwrite").parquet(s.silver_recipes_path)
interactions_s.write.mode("overwrite").parquet(s.silver_interactions_path)

print("silver written:")
print(" -", s.silver_recipes_path)
print(" -", s.silver_interactions_path)

In [ ]:
interactions_s = spark.read.parquet(s.silver_interactions_path)
interactions_s.printSchema()

In [ ]:
interactions_s.show(10, truncate=False)

In [ ]:
interactions_s.printSchema()
interactions_s.groupBy("rating").count().orderBy("rating").show()

In [ ]:
recipes_s.printSchema()

In [ ]:
from pyspark.sql import functions as F

recipes_s = spark.read.parquet(s.silver_recipes_path)
interactions_s = spark.read.parquet(s.silver_interactions_path)

print("recipes_s:", recipes_s.count(), "cols:", len(recipes_s.columns))
print("interactions_s:", interactions_s.count(), "cols:", len(interactions_s.columns))

# check null rates on cleaned fields
recipes_s.select(
    F.sum(F.col("ingredients_clean").isNull().cast("int")).alias("null_ingredients_clean"),
    F.sum(F.col("steps_clean").isNull().cast("int")).alias("null_steps_clean"),
    F.sum(F.col("tags_clean").isNull().cast("int")).alias("null_tags_clean"),
    F.sum(F.col("description_clean").isNull().cast("int")).alias("null_description_clean"),
).show()

interactions_s.select(
    F.sum(F.col("review_clean").isNull().cast("int")).alias("null_review_clean"),
    F.sum(F.col("rating").isNull().cast("int")).alias("null_rating"),
).show()

# confirm rating 0 removed
interactions_s.groupBy("rating").count().orderBy("rating").show(10)

In [ ]:
print(s.gold_reviews_path)

In [ ]:
from src.spark.ingestion.spark.transforms.merge_spark import build_gold_recipes, build_gold_reviews
from pyspark import StorageLevel

recipes_s = spark.read.parquet(s.silver_recipes_path)
interactions_s = spark.read.parquet(s.silver_interactions_path)

gold_reviews = build_gold_reviews(interactions_s)
gold_recipes = build_gold_recipes(recipes_s)

gold_reviews.write.mode("overwrite").parquet(s.gold_reviews_path)
gold_recipes.write.mode("overwrite").parquet(s.gold_recipe_path)

In [ ]:
from src.spark.ingestion.spark.transforms.merge_spark import build_gold_ve

gold_ve = build_gold_ve(gold_recipes, gold_reviews)

gold_ve.coalesce(32).write.mode("overwrite").parquet(s.gold_ve_path)
print("Wrote gold_ve")

In [ ]:
gold_reviews.schema

In [ ]:
# Row counts and column count review
print("DF SHAPE")
print("VE:", gold_ve.count(), "cols:", len(gold_ve.columns))

# Like distribution
print(40 * "-")
print("\nLIKE DISTRIBUTION")
gold_ve.groupBy("liked").count().show()

tot = gold_ve.count()
pos = gold_ve.filter(F.col("liked") == 1).count()
pct_pos = pos / tot * 100
print(f"Positive class (liked=1) count: {pos} / {tot} ({pct_pos:.2f}%)")

# Rating distribution
print(40 * "-")
print("\nRATING DISTRIBUTION")
gold_ve.groupBy("rating").count().orderBy("rating").show()

# Top reviewed recipes
print(40 * "-")
print("\nTOP 10 RECIPES BY REVIEW COUNT")
top10 = (
    gold_ve.groupBy("recipe_id", "name")
         .agg(
             F.count("*").alias("n_reviews"),
             F.round(F.avg("rating"), 3).alias("avg_rating"),
             F.sum(F.col("liked").cast("int")).alias("n_liked"),
         )
         .orderBy(F.desc("n_reviews"), F.desc("avg_rating"))
         .limit(10)
)
top10.show(truncate=60)


# Review length distribution (chars)
print(40 * "-")
print("\nREVIEW LENGTH (chars) SUMMARY")
review_len = gold_ve.withColumn("review_len", F.length(F.coalesce(F.col("review_clean"), F.lit(""))))

review_len.select(
    F.min("review_len").alias("min_len"),
    F.round(F.avg("review_len"), 2).alias("avg_len"),
    F.max("review_len").alias("max_len"),
).show()

# Percentiles (more informative than min/max)
print(40 * "-")
print("\nREVIEW LENGTH (chars) PERCENTILES")
review_len.selectExpr("percentile_approx(review_len, array(0.5, 0.9, 0.95, 0.99)) as p").show(truncate=False)

# Null counts (VE required cols)
print(40 * "-")
print("\nNULL COUNTS (VE REQUIRED)")
required_ve = [
    "user_id","recipe_id","name","rating","liked","date","review_clean",
    "ingredients_clean","steps_clean","tags_clean","description_clean",
    "minutes","n_steps","n_ingredients",
    "calories","fat","sugar","sodium","protein","saturated_fat","carbs",
]
existing_required_ve = [c for c in required_ve if c in gold_ve.columns]

gold_ve.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}") for c in existing_required_ve]
).show(truncate=False)

In [ ]:
from src.spark.labeling.zero_shot import ZeroShotConfig, prepare_zero_shot_input_from_gold, run_zero_shot_incremental_with_checkpoints


# Updated Config for the overnight marathon
cfg_overnight = ZeroShotConfig(
    taxonomy_version="v1",
    min_tokens=15,
    sample_n=20000, # Set this to your desired total target
    batch_size=4,   # Dropped slightly to be kinder to CPU thermal throttling overnight
    sample_seed=42, # Keep seed same to ensure the same 'random' order
)

# 1. Prepare input (this generates the full 'target' list)
inp = prepare_zero_shot_input_from_gold(
    spark=spark,
    gold_reviews_path=s.gold_reviews_path,  
    cfg=cfg_overnight,
    processed_dir=s.processed_dir,
)

# 2. Run with the incremental helper
print("Starting incremental zero-shot run...")
out = run_zero_shot_incremental_with_checkpoints(inp, cfg_overnight, s.processed_dir)

In [ ]:
from src.spark.labeling.zero_shot import attach_zero_shot_labels_to_gold
labeled = attach_zero_shot_labels_to_gold(spark, s.gold_reviews_path, s.processed_dir, cfg)
print("labeled parquet:", labeled)

In [ ]:
df_labeled = spark.read.parquet(labeled)
print("Labeled DF schema:")
df_labeled.printSchema()
print("Labeled DF sample:")
df_labeled.show(10, truncate=False)

In [ ]:
import pandas as pd
import json
from collections import Counter, defaultdict

from src.spark.labeling.taxonomy import get_taxonomy

def run_zs_diagnostics_pandas(zs_out_path, taxonomy_version="v1", top_k_tags=20):
    pdf = pd.read_parquet(zs_out_path)
    tax = get_taxonomy(taxonomy_version)
    pol_map = {tid: t.polarity for tid, t in tax.items()}

    def normalize_labels(x):
        if x is None:
            return []
        if isinstance(x, float) and pd.isna(x):
            return []
        if isinstance(x, str):
            return [x] if x else []
        if hasattr(x, "tolist"):
            x = x.tolist()
        if isinstance(x, (list, tuple, set)):
            return [t for t in x if t is not None and not (isinstance(t, float) and pd.isna(t))]
        return []

    n = len(pdf)
    print("n_reviews:", n)
    print("avg_labels_per_review:", round(pdf["zs_num_labels"].mean(), 3))
    print("median_labels_per_review:", pdf["zs_num_labels"].median())
    print("fallback_rate:", round(pdf["zs_used_fallback"].mean(), 4))
    print("avg_max_score:", round(pdf["zs_max_score"].mean(), 4))

    # per-tag prevalence
    tag_ctr = Counter()
    for labels in pdf["zs_labels"]:
        tag_ctr.update(normalize_labels(labels))
    print(f"\nTop {top_k_tags} tags (% of reviews):")
    for tid, cnt in tag_ctr.most_common(top_k_tags):
        print(f"{tid:28s} {cnt/max(n, 1):.3f}  ({pol_map.get(tid,'?')})")

    # polarity distribution over ASSIGNED labels
    pol_ctr = Counter()
    for labels in pdf["zs_labels"]:
        for tid in normalize_labels(labels):
            pol_ctr[pol_map.get(tid, "unknown")] += 1
    total_assigned = sum(pol_ctr.values()) or 1
    print("\nPolarity distribution over assigned labels:")
    for pol in ["positive", "negative", "neutral", "unknown"]:
        print(f"{pol:10s} {pol_ctr[pol]/total_assigned:.3f}")

    # % reviews with >=1 tag of each polarity
    def has_pol(labels, pol):
        labels = normalize_labels(labels)
        return any(pol_map.get(t) == pol for t in labels)

    print("\nReview-level polarity presence:")
    print("pct_reviews_with_pos:", round(pdf["zs_labels"].apply(lambda x: has_pol(x, "positive")).mean(), 3))
    print("pct_reviews_with_neg:", round(pdf["zs_labels"].apply(lambda x: has_pol(x, "negative")).mean(), 3))
    print("pct_reviews_with_neu:", round(pdf["zs_labels"].apply(lambda x: has_pol(x, "neutral")).mean(), 3))

    # score means by polarity (all tag scores, not just assigned)
    pol_score_sum = defaultdict(float)
    pol_score_n = defaultdict(int)
    for s in pdf["zs_scores_json"]:
        if isinstance(s, dict):
            scores = s
        elif isinstance(s, str) and s.strip():
            scores = json.loads(s)
        else:
            scores = {}
        for tid, sc in scores.items():
            pol = pol_map.get(tid, "unknown")
            pol_score_sum[pol] += float(sc)
            pol_score_n[pol] += 1

    print("\nAverage raw score by polarity (all tag scores):")
    for pol in ["positive", "negative", "neutral", "unknown"]:
        if pol_score_n[pol]:
            print(f"{pol:10s} {pol_score_sum[pol]/pol_score_n[pol]:.4f}")

# Usage:
# run_zs_diagnostics_pandas("/home/iauger/.../zs_output.parquet", taxonomy_version="v1", top_k_tags=20)

run_zs_diagnostics_pandas(labeled, taxonomy_version=cfg.taxonomy_version, top_k_tags=20)